# Matcha Career AI — IndoBERT Intent Classifier Fine-Tuning

Notebook ini digunakan untuk melakukan *fine-tuning* model Transformer **IndoBERT** (`indobenchmark/indobert-base-p1` dari Hugging Face) pada dataset klasifikasi intent Bahasa Indonesia untuk sistem chatbot Matcha Career.

## Persiapan Training:
1. Aktifkan GPU Colab: Hubungkan ke runtime GPU (**Runtime** -> **Change runtime type** -> **T4 GPU** atau **A100 GPU**).
2. Unggah file `intent_dataset.csv` yang di-generate sebelumnya ke panel berkas (files sidebar) di Colab.

### 1. Instalasi Pustaka Pendukung

In [ ]:
!pip install -q transformers[torch] datasets accelerate scikit-learn pandas

### 2. Memuat Dataset dan Eksplorasi

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Muat file CSV yang diupload
df = pd.read_csv('intent_dataset.csv')
print("Total data:", len(df))
print(df['label'].value_counts())
df.head()

### 3. Pemetaan Label & Encoding

Mengubah label teks menjadi ID numerik (0-7).

In [ ]:
labels = [
    "CAREER_EXPLORATION", "SKILL_INQUIRY", "RESOURCE_REQUEST", "CONSTRAINT_UPDATE",
    "PUSH_BACK", "CONFIRMATION", "CV_REVIEW", "LINKEDIN_REVIEW"
]
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}

df['label_id'] = df['label'].map(label2id)
df = df.rename(columns={'text': 'text', 'label_id': 'label'})

# Split dataset menjadi Train (85%) dan Evaluation (15%)
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

### 4. Setup Tokenizer IndoBERT

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

### 5. Memuat Model IndoBERT Klasifikasi

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

### 6. Matriks Evaluasi (Akurasi & F1-Score)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

### 7. Konfigurasi Pelatihan (Training Arguments) & Fine-Tuning

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Jalankan training
trainer.train()

### 8. Evaluasi Akhir Model

In [ ]:
eval_results = trainer.evaluate()
print("Hasil Evaluasi:", eval_results)

### 9. Menyimpan Model dan Membuat Zip untuk Download

In [ ]:
# Simpan model lokal
model.save_pretrained("./indobert_intent_model")
tokenizer.save_pretrained("./indobert_intent_model")

# Zip folder model untuk di-download
!zip -r indobert_intent_model.zip ./indobert_intent_model

print("\nSELESAI! Silakan unduh file 'indobert_intent_model.zip' di sidebar Files Colab Anda.")